# 01 — Data Exploration

Explore historical funding rate data across Binance, Bybit, and OKX.

**Prerequisites:** Run `python -m data.downloader --config config/default.yaml` first.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.downloader import FundingRateDownloader
from data.spot_prices import OHLCVDownloader

In [ ]:
# Load cached funding rate data
funding = FundingRateDownloader.load_cached_data()
print(f"Total records: {len(funding)}")
print(f"Exchanges: {funding['exchange'].unique()}")
print(f"Symbols: {funding['symbol'].nunique()}")
print(f"Date range: {funding['timestamp'].min()} → {funding['timestamp'].max()}")
funding.head()

In [ ]:
# Funding rate distribution by exchange
funding['annualised'] = funding['funding_rate'] * 3 * 365
fig = px.histogram(
    funding, x='annualised', color='exchange',
    title='Funding Rate Distribution (Annualised)',
    nbins=100, barmode='overlay', opacity=0.7,
    labels={'annualised': 'Annualised Funding Rate'}
)
fig.show()

In [ ]:
# Top 10 highest average funding rate pairs
avg_rates = (
    funding.groupby(['symbol', 'exchange'])['annualised']
    .mean()
    .reset_index()
    .sort_values('annualised', ascending=False)
    .head(20)
)
fig = px.bar(
    avg_rates, x='symbol', y='annualised', color='exchange',
    title='Top 20 Pairs by Average Annualised Funding Rate',
    barmode='group'
)
fig.show()

In [ ]:
# BTC funding rate over time
btc = funding[funding['symbol'] == 'BTC/USDT:USDT'].copy()
fig = px.line(
    btc, x='timestamp', y='annualised', color='exchange',
    title='BTC/USDT Funding Rate Over Time (Annualised)'
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.show()

In [ ]:
# Cross-exchange funding rate correlation
btc_pivot = btc.pivot_table(
    index='timestamp', columns='exchange', values='funding_rate'
).dropna()
print('Cross-exchange correlation:')
btc_pivot.corr()